# Model 3: Ensemble Risk Score Aggregator

**Goal:** Combine outputs from Model 1 (depression/PTSD severity from semantic+acoustic) and Model 2 (stress level from acoustic) into a single **Final Mental Health Risk Score** per participant.

**Strategy:**
- Model 1 (trained on DAIC) produces `depression_score` and `ptsd_score` for each DAIC participant.
- Model 2 (trained on StressID) is applied to DAIC participants' acoustic features to produce a `stress_score` — enabled because both datasets share the same 91 acoustic features.
- The three scores are fused into a `final_risk_score` using three aggregation methods:
  1. **Weighted Linear Combination** (interpretable, tunable weights)
  2. **Stacking** — a meta-learner trained on the three scores predicts the binary depression/PTSD label
  3. **Max-score Fusion** (conservative: flag if any score is elevated)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay,
    roc_auc_score, f1_score, roc_curve, auc
)

SEED = 42
np.random.seed(SEED)

## 1. Load Data and Re-train Model 2 on StressID

In [ ]:
# ── Load DAIC data ──
daic_feats = pd.read_csv('daic_features_v4.csv')
daic_labels = pd.read_csv('data/edaic/labels/detailed_labels.csv')
model1_out = pd.read_csv('data/edaic/labels/model1_outputs.csv')

# Merge DAIC features with labels
daic = daic_feats.merge(daic_labels, left_on='patient_id', right_on='Participant', how='inner')
PHQ_ITEMS = ['PHQ8_1_NoInterest','PHQ8_2_Depressed','PHQ8_3_Sleep','PHQ8_4_Tired',
             'PHQ8_5_Appetite','PHQ8_6_Failure','PHQ8_7_Concentration','PHQ8_8_Psychomotor']
PCL_ITEMS = [c for c in daic_labels.columns if c.startswith('PCL-C_')]
daic['PHQ8_total']        = daic[PHQ_ITEMS].sum(axis=1)
daic['PCL_total']         = daic[PCL_ITEMS].sum(axis=1)
daic['Depression_binary'] = (daic['PHQ8_total'] >= 10).astype(int)
daic['PTSD_binary']       = (daic['PCL_total']  >= 44).astype(int)

# ── Load StressID data ──
stress_acoustic = pd.read_csv('data/stressid/features/stressid_features.csv')
stress_physio   = pd.read_csv('data/stressid/features/stressid_physio_features.csv')
stress_labels   = pd.read_csv('data/StressID Dataset/labels.csv')
stress_labels[['subject_id','task']] = stress_labels['subject/task'].str.split('_', n=1, expand=True)
stress_labels = stress_labels.drop(columns=['subject/task'])
TASKS = {'Counting1','Counting2','Counting3','Math','Reading','Speaking','Stroop'}
stress_labels = stress_labels[stress_labels['task'].isin(TASKS)]

stress_df = stress_acoustic.merge(stress_labels, on=['subject_id','task'], how='inner')

print(f'DAIC participants:    {len(daic)}')
print(f'StressID samples:     {len(stress_df)} ({stress_df["subject_id"].nunique()} subjects × 7 tasks)')

In [ ]:
# ── Acoustic feature columns shared between DAIC and StressID ──
STRESSID_ACOUSTIC = [c for c in stress_acoustic.columns if c not in ('subject_id','task')]

# Train Model 2 (acoustic → binary stress) on full StressID dataset
X_stress_train = stress_df[STRESSID_ACOUSTIC].values
y_stress_train = stress_df['binary-stress'].values.astype(int)

model2_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     RandomForestClassifier(n_estimators=200, random_state=SEED))
])
model2_pipe.fit(X_stress_train, y_stress_train)

# Apply to DAIC acoustic features to get stress_score for each DAIC participant
X_daic_acoustic = daic[STRESSID_ACOUSTIC].values
daic['stress_score'] = model2_pipe.predict_proba(X_daic_acoustic)[:, 1]

print('Stress scores for DAIC participants:')
print(daic['stress_score'].describe().round(3))

## 2. Assemble the Three Scores Per Participant

In [ ]:
# Merge model1_outputs with live stress_score
scores = model1_out.merge(
    daic[['patient_id','stress_score','Depression_binary','PTSD_binary']],
    on='patient_id', how='inner'
)

print(f'Participants with all three scores: {len(scores)}')
print(scores[['patient_id','depression_score','ptsd_score','stress_score',
              'Depression_binary','PTSD_binary']].head(8).to_string(index=False))

In [ ]:
# Score distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (col, title, color) in zip(axes, [
    ('depression_score', 'Depression Score\n(Model 1, PHQ-8 normalised)', 'steelblue'),
    ('ptsd_score',       'PTSD Score\n(Model 1, PCL-C normalised)',       'darkorange'),
    ('stress_score',     'Stress Score\n(Model 2, acoustic)',              'seagreen'),
]):
    ax.hist(scores[col], bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(scores[col].mean(), color='black', linestyle='--', linewidth=1, label=f'Mean={scores[col].mean():.2f}')
    ax.set_xlabel('Score (0–1)')
    ax.set_ylabel('Count')
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('score_distributions_all3.png', dpi=150)
plt.show()

# Pairplot / scatter matrix of scores
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pairs = [('depression_score','ptsd_score'), ('depression_score','stress_score'), ('ptsd_score','stress_score')]
for ax, (x, y) in zip(axes, pairs):
    dep_color = scores['Depression_binary'].map({0:'steelblue', 1:'tomato'})
    ax.scatter(scores[x], scores[y], c=dep_color, alpha=0.6, s=20)
    ax.set_xlabel(x.replace('_score','').title())
    ax.set_ylabel(y.replace('_score','').title())
    ax.set_title(f'{x.split("_")[0].title()} vs {y.split("_")[0].title()}')
    from matplotlib.lines import Line2D
    ax.legend(handles=[
        Line2D([0],[0], marker='o', color='w', markerfacecolor='tomato', label='Depressed'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor='steelblue', label='Not depressed'),
    ], fontsize=8)

plt.tight_layout()
plt.savefig('score_scatter.png', dpi=150)
plt.show()

## 3. Method A — Weighted Linear Combination

$$\text{final\_risk} = w_1 \cdot \text{depression\_score} + w_2 \cdot \text{ptsd\_score} + w_3 \cdot \text{stress\_score}$$

We try three weight schemes and evaluate against the binary Depression label:

In [ ]:
weight_schemes = {
    'Equal (1/3 each)':       (1/3, 1/3, 1/3),
    'Depression-heavy (0.5/0.3/0.2)': (0.5, 0.3, 0.2),
    'PTSD-heavy (0.3/0.5/0.2)':       (0.3, 0.5, 0.2),
    'Stress-heavy (0.2/0.3/0.5)':     (0.2, 0.3, 0.5),
}

# Use Logistic Regression-calibrated weights (optimise w on training split)
train_scores = scores[scores['split'] == 'train']
dev_scores   = scores[scores['split'] == 'dev']

X_score_train = train_scores[['depression_score','ptsd_score','stress_score']].values
X_score_dev   = dev_scores[['depression_score','ptsd_score','stress_score']].values
y_dep_train = train_scores['Depression_binary'].values
y_dep_dev   = dev_scores['Depression_binary'].values

print('── Weighted Linear Combination (threshold=0.5, evaluated on dev set) ──')
linear_results = []
for name, (w1, w2, w3) in weight_schemes.items():
    final_train = w1*train_scores['depression_score'] + w2*train_scores['ptsd_score'] + w3*train_scores['stress_score']
    final_dev   = w1*dev_scores['depression_score']   + w2*dev_scores['ptsd_score']   + w3*dev_scores['stress_score']
    # Threshold at 0.5
    pred = (final_dev >= 0.5).astype(int)
    acc = np.mean(pred == y_dep_dev)
    f1  = f1_score(y_dep_dev, pred, average='weighted')
    auc_score = roc_auc_score(y_dep_dev, final_dev)
    print(f'  {name}: Acc={acc:.3f}  F1={f1:.3f}  AUC={auc_score:.3f}')
    linear_results.append({'weights': name, 'accuracy': acc, 'f1': f1, 'auc': auc_score,
                           'final_dev': final_dev.values})

In [ ]:
# ROC curves for each weight scheme
fig, ax = plt.subplots(figsize=(7, 6))
colors = ['steelblue','darkorange','seagreen','purple']
for r, color in zip(linear_results, colors):
    fpr, tpr, _ = roc_curve(y_dep_dev, r['final_dev'])
    ax.plot(fpr, tpr, color=color, label=f"{r['weights']} (AUC={r['auc']:.3f})")
ax.plot([0,1],[0,1],'k--', linewidth=0.8)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Weighted Fusion (Depression label)')
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig('roc_weighted_fusion.png', dpi=150)
plt.show()

## 4. Method B — Logistic Regression Stacking (Meta-Learner)

Train a Logistic Regression on top of the three scores to learn the optimal linear combination from data. Uses 5-fold cross-validation to prevent overfitting on the small training set.

In [ ]:
X_all_scores = scores[['depression_score','ptsd_score','stress_score']].values
y_dep_all    = scores['Depression_binary'].values
y_ptsd_all   = scores['PTSD_binary'].values

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

meta_models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED),
    'RandomForest':       RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED),
}

print('── Stacking Meta-Learner (5-fold CV on all 155 participants) ──')
stack_results = []
for target_name, y_target in [('Depression', y_dep_all), ('PTSD', y_ptsd_all)]:
    print(f'\n  Target: {target_name}')
    for mname, model in meta_models.items():
        pipe = Pipeline([('scaler', StandardScaler()), ('clf', model)])
        y_pred = cross_val_predict(pipe, X_all_scores, y_target, cv=cv5)
        y_prob = cross_val_predict(pipe, X_all_scores, y_target, cv=cv5, method='predict_proba')[:, 1]
        acc  = np.mean(y_pred == y_target)
        f1   = f1_score(y_target, y_pred, average='weighted')
        auc_score = roc_auc_score(y_target, y_prob)
        print(f'    {mname}: Acc={acc:.3f}  F1={f1:.3f}  AUC={auc_score:.3f}')
        stack_results.append({'target': target_name, 'model': mname,
                              'accuracy': acc, 'f1': f1, 'auc': auc_score,
                              'y_pred': y_pred, 'y_prob': y_prob, 'y_true': y_target})

In [ ]:
# Confusion matrices and ROC for stacking
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for row, target in enumerate(['Depression', 'PTSD']):
    target_results = [r for r in stack_results if r['target'] == target]
    class_names = ['Negative', 'Positive']

    for col, r in enumerate(target_results[:2]):
        ConfusionMatrixDisplay.from_predictions(
            r['y_true'], r['y_pred'],
            display_labels=class_names, ax=axes[row, col]
        )
        axes[row, col].set_title(f'{target} — {r["model"]}\nAcc={r["accuracy"]:.3f}  AUC={r["auc"]:.3f}')

plt.tight_layout()
plt.savefig('stacking_confusion.png', dpi=150)
plt.show()

# ROC curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, target in zip(axes, ['Depression', 'PTSD']):
    target_results = [r for r in stack_results if r['target'] == target]
    for r, color in zip(target_results, ['steelblue','darkorange']):
        fpr, tpr, _ = roc_curve(r['y_true'], r['y_prob'])
        ax.plot(fpr, tpr, color=color, label=f"{r['model']} (AUC={r['auc']:.3f})")
    ax.plot([0,1],[0,1],'k--', linewidth=0.8)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC — Stacking ({target})')
    ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('stacking_roc.png', dpi=150)
plt.show()

## 5. Method C — Max-Score Fusion (Conservative Screening)

Flag a participant as at-risk if **any** individual score exceeds a threshold. Optimises recall — appropriate for clinical screening where missing a case is more costly than a false alarm.

In [ ]:
scores['max_score'] = scores[['depression_score','ptsd_score','stress_score']].max(axis=1)

# Sweep threshold on training set, evaluate on dev set
train_s = scores[scores['split'] == 'train']
dev_s   = scores[scores['split'] == 'dev']

best_thresh, best_f1 = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.05):
    pred_train = (train_s['max_score'] >= thresh).astype(int)
    f = f1_score(train_s['Depression_binary'], pred_train, average='weighted')
    if f > best_f1:
        best_f1, best_thresh = f, thresh

pred_dev = (dev_s['max_score'] >= best_thresh).astype(int)
acc  = np.mean(pred_dev.values == dev_s['Depression_binary'].values)
f1   = f1_score(dev_s['Depression_binary'], pred_dev, average='weighted')
auc_s = roc_auc_score(dev_s['Depression_binary'], dev_s['max_score'])
print(f'Max-Score Fusion (threshold={best_thresh:.2f}, dev set):')
print(f'  Acc={acc:.3f}  F1={f1:.3f}  AUC={auc_s:.3f}')
print(classification_report(dev_s['Depression_binary'], pred_dev,
                            target_names=['Not Depressed', 'Depressed']))

## 6. Learned Meta-Weights (Logistic Regression Coefficients)

Inspect which score the stacking model weights most heavily — this tells us the relative importance of depression, PTSD, and stress signals.

In [ ]:
score_names = ['depression_score', 'ptsd_score', 'stress_score']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (target_name, y_target) in zip(axes, [('Depression', y_dep_all), ('PTSD', y_ptsd_all)]):
    lr = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED))
    ])
    lr.fit(X_all_scores, y_target)
    coefs = lr.named_steps['clf'].coef_[0]
    colors = ['steelblue' if c >= 0 else 'tomato' for c in coefs]
    ax.bar(score_names, coefs, color=colors)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Logistic Regression Coefficient')
    ax.set_title(f'Meta-Learner Weights — {target_name}')
    ax.set_xticklabels([s.replace('_score','') for s in score_names])
    for i, (name, val) in enumerate(zip(score_names, coefs)):
        ax.text(i, val + 0.01 * np.sign(val), f'{val:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('meta_weights.png', dpi=150)
plt.show()

## 7. Final Risk Score and Risk Tier Assignment

Compute the final risk score using the best-performing stacking method (Logistic Regression) and assign participants to risk tiers.

In [ ]:
# Fit final meta-learner on training split, score all participants
final_meta = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED))
])
final_meta.fit(X_score_train, y_dep_train)

scores['final_risk_score'] = final_meta.predict_proba(X_all_scores)[:, 1]

# Risk tiers based on quartiles of training distribution
q33 = np.percentile(final_meta.predict_proba(X_score_train)[:, 1], 33)
q66 = np.percentile(final_meta.predict_proba(X_score_train)[:, 1], 66)

def assign_tier(score):
    if score < q33:  return 'Low Risk'
    elif score < q66: return 'Moderate Risk'
    else:             return 'High Risk'

scores['risk_tier'] = scores['final_risk_score'].apply(assign_tier)

print('Risk tier distribution:')
print(scores['risk_tier'].value_counts())
print()
print('Depression rate by tier:')
print(scores.groupby('risk_tier')['Depression_binary'].mean().round(3))

In [ ]:
# Visualise risk tier vs. true labels
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

tier_order = ['Low Risk', 'Moderate Risk', 'High Risk']
tier_colors = {'Low Risk': 'seagreen', 'Moderate Risk': 'gold', 'High Risk': 'tomato'}

# Final risk score distribution by true label
for val, lbl, color in [(0,'Not Depressed','steelblue'), (1,'Depressed','tomato')]:
    sub = scores[scores['Depression_binary'] == val]
    axes[0].hist(sub['final_risk_score'], bins=15, alpha=0.6, label=lbl, color=color)
axes[0].set_xlabel('Final Risk Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Final Risk Score Distribution by True Label')
axes[0].legend()

# Depression rate per risk tier
tier_stats = scores.groupby('risk_tier')['Depression_binary'].agg(['mean','count']).reindex(tier_order)
bars = axes[1].bar(tier_order, tier_stats['mean'],
                   color=[tier_colors[t] for t in tier_order], edgecolor='white')
axes[1].set_ylabel('Proportion Depressed (PHQ-8 ≥ 10)')
axes[1].set_title('Depression Rate by Risk Tier')
axes[1].set_ylim(0, 1)
for bar, (tier, row) in zip(bars, tier_stats.iterrows()):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.02,
                 f"{row['mean']:.0%}\n(n={int(row['count'])})",
                 ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('risk_tier_analysis.png', dpi=150)
plt.show()

## 8. Full Pipeline Summary and Export

In [ ]:
# Summary table
print('=== METHOD COMPARISON (Depression label, Dev Set or 5-fold CV) ===')
summary = [
    {'Method': 'Model 1 alone (Random Forest)',  'AUC': 0.643, 'F1': 0.656, 'Note': 'Dev set'},
    {'Method': 'Model 2 alone (acoustic→stress)', 'AUC': 0.632, 'F1': 0.632, 'Note': 'Dev set (approx)'},
]
for r in linear_results:
    summary.append({'Method': f'Linear fusion — {r["weights"]}', 'AUC': round(r['auc'],3),
                    'F1': round(r['f1'],3), 'Note': 'Dev set'})
for r in stack_results:
    if r['target'] == 'Depression':
        summary.append({'Method': f'Stacking — {r["model"]}', 'AUC': round(r['auc'],3),
                        'F1': round(r['f1'],3), 'Note': '5-fold CV'})
summary.append({'Method': f'Max-score fusion (thr={best_thresh:.2f})',
                'AUC': round(auc_s,3), 'F1': round(f1,3), 'Note': 'Dev set'})

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

In [ ]:
# Export final risk scores
final_output = scores[[
    'patient_id', 'split',
    'depression_score', 'ptsd_score', 'stress_score',
    'final_risk_score', 'risk_tier',
    'Depression_binary', 'PTSD_binary',
    'PHQ8_true', 'PCL_true'
]].copy()

final_output.to_csv('data/edaic/labels/final_risk_scores.csv', index=False)
print(f'Final risk scores saved to data/edaic/labels/final_risk_scores.csv')
print(f'Shape: {final_output.shape}')
print()
print(final_output.head(10).to_string(index=False))